In [1]:
import pandas as pd
import json
import glob
import os
import re

from functools import reduce

In [2]:
triples = pd.read_excel("../../outputs/clean_outputs/triples_ISCO_ESCO_matches.xlsx")

merge = pd.read_excel("../../outputs/clean_outputs/triples_clean.xlsx")[["Unnamed: 0", "id"]]
merge.rename(columns={"id" : "humanjobid"})

triples = pd.merge(triples, merge, left_on="id", right_on="Unnamed: 0", how="left")

In [3]:
triples.head()

,Unnamed: 0_x,id_x,ISCO gemma semi-structured,ESCO gemma semi-structured,ISCO gemma structured,ESCO gemma structured,ISCO gemma unstructured,ESCO gemma unstructured,ISCO llama semi-structured,ESCO llama semi-structured,...,ESCO llama unstructured,ISCO qwen semi-structured,ESCO qwen semi-structured,ISCO qwen structured,ESCO qwen structured,ISCO qwen unstructured,ESCO qwen unstructured,humanjobid,Unnamed: 0_y,id_y
0,5,6,"[(""medarbejdere"", ""has_isco"", ""5312"")]","[(""respond to customers' inquiries"", ""has_esco...","[(""Glade medarbejdere"", ""has_isco"", ""8321""), (...","[(""Customer service"", ""has_esco"", ""001160"")]","[(""Glade medarbejdere til et stærkt kundeservi...","[(""Customer Service"", ""has_esco"", ""001160"")]",[('Glade medarbejdere til et stærkt kundeserv...,[('Glade medarbejdere til et stærkt kundeserv...,...,"After evaluating the triples, I found the foll...","[(""Glade medarbejdere til et stærkt kundeservi...",[],[],[],[],[],1527442,6,1527454
1,6,7,"[(""meat segment"", ""has_isco"", ""8160""), (""Indus...","[(""Industry Manager"", ""has_esco"", ""000046""), (...","[(""meat segment"", ""has_isco"", ""8160""), (""Indus...","[(""Industry Manager"", ""has_esco"", ""000046""), (...","[(""Industry Manager (meat segment)"", ""has_job_...","[(""Industry Manager (meat segment)"", ""has_skil...","[('Cattle farmer', 'has_isco', 6111), ('Dairy...",Here is the list of ESCO triples:\n\n[('manage...,...,Here are the ESCO-triples for the given subjec...,[],[],"[(""meat segment"", ""has_isco"", ""8160""), (""Indus...",[],[],[],1527454,7,1527460
2,10,11,"[(""Driftsleder til Facility Management"", ""has_...","[(""Communication"", ""has_esco"", ""006552""), (""Pl...","[(""Driftsleder til Facility Management"", ""has_...","[(""Facility Management"", ""has_esco"", ""000046"")...","[(""Driftsleder til Facility Management"", ""has_...","[(""Driftsleder til Facility Management"", ""has_...","[('Facility Manager', 'has_isco', '3151'), ('...","[('Facility Manager', 'has_esco', '000046'), ...",...,Based on the provided ESCO codes and their def...,"[(""Driftsleder til Facility Management"", ""has_...","[(""facility management"", ""has_esco"", ""000046"")...",[],"[(""Building Inspections"", ""has_esco"", ""009910""...","[(""Driftsleder til Facility Management"", ""has_...","[(""knowledge_of_health_and_safety_regulations""...",1527480,11,1527482
3,11,12,"[(""Minibuschauffør"", ""has_isco"", ""5112""), (""Mi...","[(""011717"", ""has_esco"", ""operation of transpor...","[(""Minibuschauffør"", ""has_isco"", ""8321"")]","[(""011717"", ""has_esco"", ""operation of transpor...","[(""Minibuschauffør"", ""has_isco"", ""5112"")]","[(""Minibuschauffør"", ""has_esco"", ""011717""), (""...","[('Fordonshåndtering', 'has_isco', 3151), ('P...",Here is the list of ESCO triples:\n\n [('Minib...,...,Here are the ESCO triples:\n\n[('Minibuschauff...,"[(""Minibuschauffør"", ""has_isco"", ""5112"")]","[(""Minibuschauffør"", ""has_esco"", ""011717"")]","[(""Minibuschauffør"", ""has_isco"", ""5112"")]","[(""operation of transport equipment"", ""has_esc...",[],[],1527482,12,1527487
4,16,17,"[(""Environmental Impact Assessment"", ""has_isco...","[(""Environmental Impact Assessment"", ""has_esco...","[(""Senior Environmental Specialist"", ""has_isco...","[(""Senior Environmental Specialist"", ""has_esco...","[(""Senior Environmental Specialist (Aalborg)"",...","[(""Environmental Impact Assessment"", ""has_esco...","[('Senior Environmental Specialist', 'has_isc...","[('Senior Environmental Specialist', 'has_esc...",...,"[('has_job_title', 'Environmental Specialist'...",[],"[(""Environmental Assessment"", ""has_esco"", ""000...","[(""Senior Environmental Specialist"", ""has_isco...","[(""educate people about nature"", ""has_esco"", ""...","[(""Senior Environmental Specialist"", ""has_isco...","[(""proficiency in environmental data analysis""...",1527507,17,1527537


In [4]:
# 1. Get a list of all your JSON files
file_pattern = './logs_isco_esco/temporary_results_*.json' 
files = glob.glob(file_pattern)

# Step 1: Group all DataFrames by their model/prompt label
grouped_data = {}

for file in files:
    if os.path.getsize(file) == 0:
        continue

    try:
        with open(file, 'r') as f:
            data = json.load(f)
        
        temp_df = pd.DataFrame(data)
        
        # Extract label (e.g., 'qwen unstructured')
        basename = os.path.basename(file)
        name_match = re.search(r'results_(.*?)_\d', basename)
        label = name_match.group(1).rstrip('_').replace('_', ' ') if name_match else "unknown"

        if label not in grouped_data:
            grouped_data[label] = []
        
        grouped_data[label].append(temp_df)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# Step 2: For each label, "squash" multiple files into one high-density DF
final_model_dfs = []

for label, dfs in grouped_data.items():
    # Stack all files for this model vertically
    combined = pd.concat(dfs, ignore_index=True)
    
    # Sort so that if there are duplicates, we have a consistent pick 
    # (Optional: sort by a timestamp if you want the newest values to take priority)
    # combined = combined.sort_values('some_timestamp_column', ascending=False)

    # The "Squash": Group by ID and take the first non-null value for ISCO and ESCO
    # This fills gaps where one file had IDs 0-500 and another had 500-1000
    squashed = combined.groupby('id', as_index=False).first()

    # Rename to your specific format
    rename_map = {
        'ISCO': f'ISCO {label}',
        'ESCO': f'ESCO {label}'
    }
    squashed = squashed.rename(columns=rename_map)
    
    # Keep only the ID and the new model-specific columns
    cols_to_keep = ['id', f'ISCO {label}', f'ESCO {label}']
    squashed = squashed[squashed.columns.intersection(cols_to_keep)]
    
    final_model_dfs.append(squashed)

# Step 3: Horizontal merge of the distinct models
if final_model_dfs:
    main_df = final_model_dfs[0]
    for next_df in final_model_dfs[1:]:
        main_df = pd.merge(main_df, next_df, on='id', how='outer')
    
    main_df = main_df.sort_values('id').reset_index(drop=True)
    
    print(f"Final Shape: {main_df.shape}")
    print("\nSample of merged columns:")
else:
    print("No data processed.")

Final Shape: (10580, 21)

Sample of merged columns:


In [5]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10580 entries, 0 to 10579
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   id                          10580 non-null  int64 
 1   ISCO cv qwen                112 non-null    object
 2   ESCO cv qwen                112 non-null    object
 3   ISCO gemma semi-structured  10580 non-null  object
 4   ESCO gemma semi-structured  10580 non-null  object
 5   ISCO gemma structured       10580 non-null  object
 6   ESCO gemma structured       10580 non-null  object
 7   ISCO gemma unstructured     10580 non-null  object
 8   ESCO gemma unstructured     10580 non-null  object
 9   ISCO llama semi-structured  10580 non-null  object
 10  ESCO llama semi-structured  10580 non-null  object
 11  ISCO llama structured       10580 non-null  object
 12  ESCO llama structured       10580 non-null  object
 13  ISCO llama unstructured     10580 non-null  ob

In [6]:
df_int = pd.read_csv("../../../dataset/final_dataset/contacted_anon.csv")

# Only keep vacancies with at least 15 interactions
relevant_vacancies = df_int["cvid"].value_counts()[df_int["cvid"].value_counts() >= 15].index

# Filter to only relevant vacancies
df_int = df_int[df_int["cvid"].isin(relevant_vacancies.values)][["humanjobid", "cvid"]]

In [7]:
df_trips = pd.read_excel("../../outputs/clean_outputs/filtered_triples.xlsx")[["id"]]

In [8]:
rel_vacancies = set(df_int["humanjobid"].values)
len(rel_vacancies)

4979

In [9]:
main_df["humanjobid"] = df_trips

In [10]:
# main_df = main_df[main_df["humanjobid"].isin(rel_vacancies)]
main_df.head()

,id,ISCO cv qwen,ESCO cv qwen,ISCO gemma semi-structured,ESCO gemma semi-structured,ISCO gemma structured,ESCO gemma structured,ISCO gemma unstructured,ESCO gemma unstructured,ISCO llama semi-structured,...,ESCO llama structured,ISCO llama unstructured,ESCO llama unstructured,ISCO qwen semi-structured,ESCO qwen semi-structured,ISCO qwen structured,ESCO qwen structured,ISCO qwen unstructured,ESCO qwen unstructured,humanjobid
0,1,NaN,NaN,"[(""IT-administrator"", ""has_isco"", ""2522""), (""I...","[(""IT-administrator"", ""has_esco"", ""011116"")]","[(""IT-administrator"", ""has_isco"", ""2522"")]","[(""IT-administrator"", ""has_esco"", ""012179"")]","[(""IT-administrator – få indflydelse på et set...","[(""IT administration"", ""has_esco"", ""011151""), ...","([('IT-administrator', 'has_isco', '2522'), ('...",...,"[('IT-administrator', 'has_esco', '000046'), ...","[('Database Administrator', 'has_isco', '2521'...",Here are the linked triples:\n\n[('IT Administ...,"[(""IT-administrator"", ""has_isco"", ""2522"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]","[(""IT-administrator"", ""has_isco"", ""2521"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]","[(""IT-administrator"", ""has_isco"", ""1330"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]",1527392
1,2,NaN,NaN,"[(""SQE Manager"", ""has_isco"", ""7536"")]","[(""Test Execution"", ""has_esco"", ""010569""), (""D...","[(""SQE Manager"", ""has_isco"", ""7536"")]","[(""008265"", ""has_esco"", ""apply risk management...","[(""SQE Manager"", ""has_isco"", ""1420""), (""SQE Ma...","[(""Kvalitetssikring"", ""has_esco"", ""008265""), (...","[('SQE Manager', 'has_isco', '7515'), ('SQE M...",...,"After analyzing the provided triples, I have i...","[('SQE Manager', 'has_isco', '6111'), ('SQE M...",Here are the ESCO triples:\n\n [('SQE Manager'...,"[(""SQE Manager"", ""has_isco"", ""7543"")]","[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ...","[(""SQE Manager"", ""has_isco"", ""7543"")]","[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ...",[],"[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ...",1527395
2,3,NaN,NaN,"[(""Ships' Deck Officers and Pilots"", ""has_isco...","[(""012179"", ""has_esco"", ""process order forms w...",[],"[(""001160"", ""has_esco"", ""customer service"")]","[(""Kundeservice / teknisk support"", ""has_isco""...","[(""Kundeservice / teknisk support"", ""has_esco""...","[('Kundeservice', 'has_isco', '5312'), ('Kund...",...,Here is the list of ESCO triples:\n\n [('Kunde...,"[('Kundeservice', 'har', '5329'), ('Kundeserv...",Here is the list of ESCO triples:\n\n [('Kunde...,"[(""Kundeservice / teknisk support"", ""has_isco""...","[(""customer service"", ""has_esco"", ""001160"")]","[(""Kundeservice / teknisk support"", ""has_isco""...","[(""technical communication"", ""has_esco"", ""0065...","[(""Kundeservice / teknisk support"", ""has_isco""...","[(""customer service"", ""has_esco"", ""01160"")]",1527397
3,4,NaN,NaN,"[(""Retail designer"", ""REQUIRES_SKILL"", ""AutoCA...","[(""Retail designer"", ""REQUIRES_SKILL"", ""010923...",[],"[(""Retail designer"", ""has_esco"", ""001496""), (""...","[(""Retail designer med teknikken på plads"", ""h...","[(""Retail designer med teknikken på plads"", ""h...","[('Retail designer med teknikken på plads', '...",...,"[('Retail designer med teknikken på plads', '...","[('Retail designer', 'has_isco', '7513'), ('R...",Here are the ESCO triples for the given triple...,[],"[(""use creative suite software"", ""has_esco"", ""...",[],[],[],"[(""Retail Designer"", ""has_esco"", ""006381""), (""...",1527417
4,5,NaN,NaN,"[(""Supporter til Caseware"", ""has_isco"", ""5312""...","[(""003500"", ""has_esco"", ""manage packaging mate...",[],[],"[(""Customer Support"", ""has_isco"", ""4419""), (""C...","[(""Communication"", ""has_esco"", ""009910""), (""Pr...","[('Docker', 'has_isco', 7536), \n ('detail-or...",...,Here are the ESCO triples for the given subjec...,"[('Caseware', 'has_isco', 7533), ('Caseware',

In [11]:
main_df.to_excel("../../outputs/raw_outputs/ISCO_ESCO_triples.xlsx")

In [12]:
# 1. Identify all unique model/prompt combinations from the columns
# We look for columns starting with 'ISCO ' or 'ESCO '
columns = main_df.columns
model_prompt_pairs = set()

for col in columns:
    if col.startswith('ISCO '):
        model_prompt_pairs.add(col.replace('ISCO ', ''))

todo = {}

for pair in model_prompt_pairs:
    # Split "qwen unstructured" into model_name and prompt_type
    # We use rsplit to handle models that might have spaces in their name
    parts = pair.rsplit(' ', 1)
    model_name = parts[0]
    prompt_type = parts[1] if len(parts) > 1 else "default"
    
    # 2. Find IDs where ISCO or ESCO is null for this pair
    isco_col = f'ISCO {pair}'
    esco_col = f'ESCO {pair}'
    
    # Logic: An ID needs work if EITHER ISCO or ESCO is missing
    missing_mask = main_df[isco_col].isna() | main_df[esco_col].isna()
    missing_ids = main_df.loc[missing_mask, 'id'].tolist()
    
    # 3. Build the nested dictionary
    if model_name not in todo:
        todo[model_name] = {}
    
    todo[model_name][prompt_type] = missing_ids

# 4. Save to todo.json
with open('todo_ISCO.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created! Found {len(model_prompt_pairs)} model/prompt configurations.")

todo.json created! Found 10 model/prompt configurations.


In [13]:
# 1. Define the master lists based on your pipeline's needs
expected_models = ['qwen', 'gemma', 'llama'] 
# Added 'semi-structured' to match your Traceback error
expected_prompts = ['structured', 'unstructured', 'semi-structured'] 

# 2. Get the full list of all IDs (10,580 entries)
all_ids = main_df['humanjobid'].unique().tolist()

todo = {}

for model in expected_models:
    todo[model] = {}
    for prompt in expected_prompts:
        # Standardize the label used in columns: "model prompt"
        # We check both "model_prompt" and "model prompt" to be safe
        pair_label = f"{model} {prompt}"
        isco_col = f'ISCO {pair_label}'
        esco_col = f'ESCO {pair_label}'
        
        # Check if we have any existing data for this combination
        if isco_col in main_df.columns:
            # Find IDs where either ISCO or ESCO is NaN
            missing_mask = main_df[isco_col].isna() | main_df[f'ESCO {pair_label}'].isna()
            missing_ids = main_df.loc[missing_mask, 'id'].tolist()
            todo[model][prompt] = [int(i) for i in missing_ids] # Ensure they are Python ints
        else:
            # If the model/prompt is totally missing, ALL IDs are todos
            todo[model][prompt] = all_ids

# 3. Save to todo.json
with open('todo_ISCO.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created. Included {len(expected_prompts)} prompt types for {len(expected_models)} models.")

todo.json created. Included 3 prompt types for 3 models.


In [14]:
for k, v in todo.items():
    for l, p in v.items():
        print(k, l, len(p))

qwen structured 0
qwen unstructured 0
qwen semi-structured 0
gemma structured 0
gemma unstructured 0
gemma semi-structured 0
llama structured 0
llama unstructured 0
llama semi-structured 0
